In [2]:
!pip install rdkit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.1/37.1 MB 36.8 MB/s eta 0:00:00


In [6]:
from rdkit import Chem
from rdkit.Chem import DataStructs
from rdkit.Chem import rdFingerprintGenerator
import numpy as np

# ── 27 unique Smyth compounds (Smyth et al., 1969) ──────────────────────────
smyth_smiles = {
    'Acetone':              'CC(C)=O',
    'Methanol':             'CO',
    'Ethanol':              'CCO',
    'n-Propanol':           'CCCO',
    'Isopropanol':          'CC(C)O',
    'n-Butanol':            'CCCCO',
    'Ethylene glycol':      'OCCO',
    'Glycerol':             'OCC(O)CO',
    'Acetic acid':          'CC(O)=O',
    'Malonic acid':         'OC(=O)CC(=O)O',
    'Succinic acid':        'OC(=O)CCC(=O)O',
    'Ethyl acetate':        'CCOC(C)=O',
    'Diethyl ether':        'CCOCC',
    'Triethylamine':        'CCN(CC)CC',
    'Hexane':               'CCCCCC',
    'Heptane':              'CCCCCCC',
    'Octane':               'CCCCCCCC',
    'Cyclohexane':          'C1CCCCC1',
    'Cyclohexanol':         'OC1CCCCC1',
    'Benzene':              'c1ccccc1',
    'Toluene':              'Cc1ccccc1',
    'Benzaldehyde':         'O=Cc1ccccc1',
    'Acetophenone':         'CC(=O)c1ccccc1',
    'Chlorobenzene':        'Clc1ccccc1',
    'Dichloromethane':      'ClCCl',
    '1,3-Dichloropropane':  'ClCCCl',
    'Carbon tetrachloride': 'ClC(Cl)(Cl)Cl',
}

# ── 9 unique clinical drugs from 5 test cases ────────────────────────────────
clinical_smiles = {
    'Tramadol':    'OC1(c2ccccc2OC)CCCCC1CN(C)C',         # cyclohexanol core
    'Diazepam':    'CN1C(=O)CN=C(c2ccccc2)c2cc(Cl)ccc21',
    'Paracetamol': 'CC(=O)Nc1ccc(O)cc1',
    'Phenytoin':   'O=C1NC(=O)NC1(c1ccccc1)c1ccccc1',
    'Warfarin':    'CC(=O)CC(c1ccccc1)c1c(O)c2ccccc2oc1=O',
    'Ibuprofen':   'CC(C)Cc1ccc(cc1)C(C)C(=O)O',
    'Vitamin C':   'OC[C@H](O)[C@H]1OC(=O)C(O)=C1O',
    'Sertraline':  'CNC1CCC(c2ccc(Cl)c(Cl)c2)c2ccccc21',
}

# ── Fingerprint generator (non-deprecated) ───────────────────────────────────
morgan_gen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)

def get_fp(smi, name=""):
    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        print(f"  WARNING: Could not parse SMILES for {name}: {smi}")
        return None
    return morgan_gen.GetFingerprint(mol)

# ── Validate and build Smyth fingerprints ────────────────────────────────────
print("=" * 55)
print("Validating Smyth compound SMILES...")
smyth_fps = []
for name, smi in smyth_smiles.items():
    fp = get_fp(smi, name)
    if fp is not None:
        smyth_fps.append(fp)

print(f"Valid Smyth compounds: {len(smyth_fps)} / {len(smyth_smiles)}")

# ── Compute Tanimoto similarities ────────────────────────────────────────────
print("=" * 55)
print("Tanimoto Similarity: Clinical Drugs vs Smyth Compounds")
print("=" * 55)

results = {}
for drug_name, smi in clinical_smiles.items():
    fp = get_fp(smi, drug_name)
    if fp is not None:
        sims = [DataStructs.TanimotoSimilarity(fp, sfp) for sfp in smyth_fps]
        max_sim = max(sims)
        closest_smyth = list(smyth_smiles.keys())[sims.index(max_sim)]
        results[drug_name] = max_sim
        print(f"{drug_name:<15} max = {max_sim:.4f}  "
              f"(closest Smyth: {closest_smyth})")

# ── Summary ──────────────────────────────────────────────────────────────────
print("=" * 55)
all_sims = list(results.values())
print(f"Range : {min(all_sims):.4f} to {max(all_sims):.4f}")
print(f"Mean  : {np.mean(all_sims):.4f}")
print(f"Median: {np.median(all_sims):.4f}")
print("=" * 55)
print("NOTE: All values below 0.40 confirm low structural")
print("overlap between training and clinical test drugs.")

Validating Smyth compound SMILES...
Valid Smyth compounds: 27 / 27
Tanimoto Similarity: Clinical Drugs vs Smyth Compounds
Tramadol        max = 0.1458  (closest Smyth: Acetophenone)
Diazepam        max = 0.2250  (closest Smyth: Chlorobenzene)
Paracetamol     max = 0.2857  (closest Smyth: Acetophenone)
Phenytoin       max = 0.2414  (closest Smyth: Benzaldehyde)
Warfarin        max = 0.2619  (closest Smyth: Acetophenone)
Ibuprofen       max = 0.2424  (closest Smyth: Acetophenone)
Vitamin C       max = 0.2143  (closest Smyth: Glycerol)
Sertraline      max = 0.1842  (closest Smyth: Chlorobenzene)
Range : 0.1458 to 0.2857
Mean  : 0.2251
Median: 0.2332
NOTE: All values below 0.40 confirm low structural
overlap between training and clinical test drugs.
